## 4-2. トロッター分解を用いた量子ダイナミクスシミュレーション

> 移行メモ
> この節は PDF と草稿を突き合わせて移行している。図，回路図，出力図はプレースホルダに置き換えた。QURI Parts による実装例は書籍の流れに沿って掲載しているが，現時点では実行セルではなくコード例として載せている。

4.1 節で説明したように，量子系の時間発展はシュレーディンガー方程式に従うが，その厳密な計算には指数関数的に大きな計算資源が必要となる。そこで，時間発展演算子を量子回路として実装可能な形へ落とし込む基本手法として，トロッター分解が用いられる。本節では，トロッター分解の考え方を解説したうえで，イジングモデルと横磁場イジングモデルの時間発展を例に，QURI Parts を用いた実装の流れを紹介する。

### 4.2.1 トロッター分解のアルゴリズム

ハミルトニアン $H$ が複数の項の和

$$
H = \sum_{i=0}^{L-1} H_i
$$

として書けるとする。一般に各項は可換ではないため，

$$
e^{-iHt} \neq e^{-iH_0 t}e^{-iH_1 t}\cdots e^{-iH_{L-1} t}
$$

である。しかし，時間区間を細かく分割して $\Delta t=t/M$ とすると，各ステップで

$$
e^{-iH\Delta t} = e^{-i(H_0 + H_1 + \cdots + H_{L-1})\Delta t} \approx e^{-iH_0\Delta t}e^{-iH_1\Delta t}\cdots e^{-iH_{L-1}\Delta t}
$$

と近似できる。これを $M$ 回繰り返すことで，全体の時間発展を近似するのがトロッター分解の基本的な考え方である。

### イジングモデル

具体例として，1 次元周期系のイジングモデル

$$
H = J(Z_0Z_1 + Z_1Z_2 + Z_2Z_3 + Z_3Z_4 + Z_4Z_0)
$$

を考える。ここで $J$ は結合強度であり，$Z_iZ_{i+1}$ は隣接スピン間の相互作用を表す。$Z$ 方向のスピンがそろうか互い違いになるかによってエネルギーが変わり，その振る舞いが磁性のモデル化に対応する。

2 粒子の場合，必要な基本操作は

$$
e^{-iJZ_0Z_1\Delta t}
$$

であり，これは 2 つの CNOT と 1 つの $R_Z$ ゲートを用いて実装できる。すなわち，$R_{ZZ}(\theta)=e^{-i\theta Z\otimes Z/2}$ は標準的な量子回路へ分解可能である。

[図 4.1 プレースホルダ: 2 粒子イジングモデルにおける 1 タイムステップの量子回路]

[図 4.2 プレースホルダ: 横磁場イジングモデルにおける 1 タイムステップの量子回路]

以下は，5 量子ビットのイジングモデルに対して，全磁化の時間発展を QURI Parts で比較するためのコード例である。厳密な時間発展は行列指数関数を用いて構成し，トロッター分解による近似と比較する。

```python
from quri_parts.core.operator import Operator, pauli_label, get_sparse_matrix
from quri_parts.core.state import quantum_state, apply_circuit, CircuitQuantumState
from quri_parts.circuit import QuantumCircuit
from quri_parts.qulacs.estimator import create_qulacs_vector_estimator
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm

n_qubits = 5
t = 3.0
M = 300
delta = t / M
J = 1.0

magnetization_obs = Operator({
    pauli_label(f"Z{i}"): 1 / n_qubits
    for i in range(n_qubits)
})

def get_exact_time_evolution_circuit() -> QuantumCircuit:
    hamiltonian = Operator()
    for i in range(n_qubits):
        hamiltonian.add_term(pauli_label(f"Z{i} Z{(i+1)%n_qubits}"), 1)
    hamiltonian_matrix = get_sparse_matrix(hamiltonian).toarray()
    exp_h = expm(-1j * hamiltonian_matrix * delta)
    circuit = QuantumCircuit(n_qubits)
    circuit.add_UnitaryMatrix_gate(range(n_qubits), exp_h)
    return circuit

def get_trotter_time_evolution_circuit() -> QuantumCircuit:
    circuit = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        circuit.add_CNOT_gate(i, (i+1)%n_qubits)
        circuit.add_RZ_gate((i+1)%n_qubits, -2 * J * delta)
        circuit.add_CNOT_gate(i, (i+1)%n_qubits)
    return circuit
```

### 横磁場イジングモデル

次に，$x$ 軸方向の磁場を加えた横磁場イジングモデル

$$
H = J(Z_0Z_1 + Z_1Z_2 + Z_2Z_3 + Z_3Z_4 + Z_4Z_0) + h(X_0 + X_1 + X_2 + X_3 + X_4)
$$

を考える。ここで $h$ は横磁場の強さである。$ZZ$ 相互作用に加えて，各量子ビットに対する $X$ 回転が入るため，磁化は一般には保存されず，時間とともに振動する。トロッター分解では，$ZZ$ 項と $X$ 項をそれぞれ実装可能なゲート列へ分けて近似する。

横磁場イジングモデルのコード例は次のようになる。

```python
h = 3.0

def get_trotter_time_evolution_circuit() -> QuantumCircuit:
    circuit = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        circuit.add_CNOT_gate(i, (i+1)%n_qubits)
        circuit.add_RZ_gate((i+1)%n_qubits, -2 * delta)
        circuit.add_CNOT_gate(i, (i+1)%n_qubits)
        circuit.add_RX_gate(i, -2 * delta * h)
    return circuit

def run_time_evolution(initial_state, trotter_circuit, exact_circuit):
    estimator = create_qulacs_vector_estimator()
    y_trotter = [estimator(magnetization_obs, initial_state).value.real]
    y_exact = [estimator(magnetization_obs, initial_state).value.real]

    for i in range(M):
        exact_circuit_i = QuantumCircuit(n_qubits, exact_circuit.gates * (i + 1))
        trotter_circuit_i = QuantumCircuit(n_qubits, trotter_circuit.gates * (i + 1))
        exact_state_i = apply_circuit(exact_circuit_i, initial_state)
        trotter_state_i = apply_circuit(trotter_circuit_i, initial_state)
        y_exact.append(estimator(magnetization_obs, exact_state_i).value.real)
        y_trotter.append(estimator(magnetization_obs, trotter_state_i).value.real)

    return y_trotter, y_exact
```

### トロッター分解における近似エラー

元の時間発展演算子をテイラー展開した式と，トロッター分解後の積を展開した式を比較すると，各ステップで生じる誤差はハミルトニアン各項の交換子に支配されることがわかる。概念的には，各タイムステップの誤差は $O(\Delta t^2)$，全時間発展での累積誤差はおおむね $O(t^2/M)$ のスケールで振る舞う。このため，精度を高めるには時間刻みを細かくする必要があるが，そのぶん回路深さは増大する。これがトロッター分解の基本的なトレードオフである。

> 注記
> 書籍版ではこの節の後半で，厳密時間発展とトロッター近似との差のプロット例や誤差評価の式展開が続く。今回は本文の主旨と実装の流れを優先し，図の再現と長い式番号付き導出は省略した。必要なら後で数式をさらに厳密に書き戻す。